In [9]:
# ==========================================
# Titanic Classification Project (Final)
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# ------------------------------------------
# 1. Load Dataset
# ------------------------------------------

df = pd.read_csv("Titanic-Dataset.csv")

# ------------------------------------------
# 2. Data Preprocessing (No Warnings Version)
# ------------------------------------------

df.fillna({
    'Age': df['Age'].median(),
    'Embarked': df['Embarked'].mode()[0],
    'Fare': df['Fare'].median()
}, inplace=True)

# Encode categorical columns
le = LabelEncoder()
df['Sex'] = le.fit_transform(df['Sex'])
df['Embarked'] = le.fit_transform(df['Embarked'])

# Feature Selection
X = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]
y = df['Survived']

# ------------------------------------------
# 3. Train Test Split
# ------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ------------------------------------------
# 4. XGBoost Model
# ------------------------------------------

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    random_state=42,
    verbosity=0,          # Removes logs
    use_label_encoder=False
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print("===== XGBoost Classification =====")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("Classification Report:\n", classification_report(y_test, y_pred_xgb))

# Cross Validation (Stratified)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_xgb = cross_val_score(xgb_model, X, y, cv=skf, scoring='accuracy')
print("XGBoost CV Accuracy:", cv_xgb.mean())

# ------------------------------------------
# 5. LightGBM Model (Silent Version)
# ------------------------------------------

lgb_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=50,
    min_child_samples=10,
    max_depth=6,
    random_state=42,
    verbose=-1   # Removes training logs
)

lgb_model.fit(X_train, y_train)

y_pred_lgb = lgb_model.predict(X_test)

print("\n===== LightGBM Classification =====")
print("Accuracy:", accuracy_score(y_test, y_pred_lgb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lgb))
print("Classification Report:\n", classification_report(y_test, y_pred_lgb))

cv_lgb = cross_val_score(lgb_model, X, y, cv=skf, scoring='accuracy')
print("LightGBM CV Accuracy:", cv_lgb.mean())

# ------------------------------------------
# 6. Best Model Comparison
# ------------------------------------------

if cv_xgb.mean() > cv_lgb.mean():
    print("\nBest Model: XGBoost")
else:
    print("\nBest Model: LightGBM")

===== XGBoost Classification =====
Accuracy: 0.8044692737430168
Confusion Matrix:
 [[98 12]
 [23 46]]
Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.89      0.85       110
           1       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179

XGBoost CV Accuracy: 0.8405875337392505

===== LightGBM Classification =====
Accuracy: 0.7932960893854749
Confusion Matrix:
 [[96 14]
 [23 46]]
Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.87      0.84       110
           1       0.77      0.67      0.71        69

    accuracy                           0.79       179
   macro avg       0.79      0.77      0.78       179
weighted avg       0.79      0.79      0.79       179

LightGBM CV Accuracy: 0.8327349193396522

Best Model

In [10]:
# ==========================================
# Titanic Regression Project (Final)
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# ------------------------------------------
# 1. Load Dataset
# ------------------------------------------

df = pd.read_csv("Titanic-Dataset.csv")

# ------------------------------------------
# 2. Data Preprocessing
# ------------------------------------------

df = df.copy()

df.fillna({
    'Age': df['Age'].median(),
    'Embarked': df['Embarked'].mode()[0],
    'Fare': df['Fare'].median()
}, inplace=True)

# Encode categorical features
le = LabelEncoder()
df['Sex'] = le.fit_transform(df['Sex'])
df['Embarked'] = le.fit_transform(df['Embarked'])

# ------------------------------------------
# 3. Features & Target
# ------------------------------------------

X = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Embarked']]
y = df['Fare']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ------------------------------------------
# 4. XGBoost Regression
# ------------------------------------------

xgb_reg = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    verbosity=0
)

xgb_reg.fit(X_train, y_train)
y_pred_xgb = xgb_reg.predict(X_test)

print("===== XGBoost Regression =====")
print("MSE:", mean_squared_error(y_test, y_pred_xgb))
print("R2 Score:", r2_score(y_test, y_pred_xgb))

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_xgb = cross_val_score(xgb_reg, X, y, cv=kf, scoring='r2')
print("XGBoost CV R2:", cv_xgb.mean())

# ------------------------------------------
# 5. LightGBM Regression
# ------------------------------------------

lgb_reg = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=50,
    random_state=42,
    verbose=-1
)

lgb_reg.fit(X_train, y_train)
y_pred_lgb = lgb_reg.predict(X_test)

print("\n===== LightGBM Regression =====")
print("MSE:", mean_squared_error(y_test, y_pred_lgb))
print("R2 Score:", r2_score(y_test, y_pred_lgb))

cv_lgb = cross_val_score(lgb_reg, X, y, cv=kf, scoring='r2')
print("LightGBM CV R2:", cv_lgb.mean())

# ------------------------------------------
# 6. Best Model
# ------------------------------------------

if cv_xgb.mean() > cv_lgb.mean():
    print("\nBest Regression Model: XGBoost")
else:
    print("\nBest Regression Model: LightGBM")

===== XGBoost Regression =====
MSE: 1764.370690058282
R2 Score: -0.14019446694240179
XGBoost CV R2: 0.3335293571614992

===== LightGBM Regression =====
MSE: 1420.5717237582583
R2 Score: 0.08197975150534142
LightGBM CV R2: 0.37443169504485835

Best Regression Model: LightGBM
